# Constructing a RAG System and Enhancing it with HyDE

This notebook builds a baseline RAG workflow and improves retrieval quality with HyDE for question answering over a technical document.

## Recommended Hardware

This notebook can run on the following hardware or remote resources

✅ AMD Instinct™ Accelerators  
✅ AMD Radeon™ RX/PRO Graphics Cards  
✅ AMD EPYC™ Processors  
✅ AMD Ryzen™ (AI) Processors  

[![Open in AMD Developer Cloud](https://img.shields.io/badge/Open_in_AMD_Developer_Cloud-000000?logo=amd&logoSize=auto)](https://amd-ai-academy.com/github/AMDResearch/aup-ai-tutorials/blob/main/rag/03.rag-pipeline-HyDE.ipynb)  


## Software Environment

Install ROCm on your system

| Linux | Windows |
|-------|---------|
| [Install PyTorch](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/install/quick-start.html) | [PyTorch on Windows](https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/windows/install-pytorch.html)|
| [Install Docker container](https://amdresearch.github.io/aup-ai-tutorials//env/env-gpu.html) | |

## Goals

- Build a basic RAG pipeline using a real technical document
- Compare standard retrieval with HyDE-based retrieval
- Understand how prompt design affects retrieval quality
- Read and explain LangChain LangChain Expression Language pipelines built with `|` and `RunnablePassthrough`

### Install Dependencies

Install the package dependencies needed for this notebook or series of notebooks.

First, get the `aup_config.py` script locally if needed. Then install the dependencies (`aup_setup()`). This step may take a few minutes and only needs to be done once.

In [ ]:
![ -f aup_config.py ] || wget https://raw.githubusercontent.com/AMDResearch/aup-ai-tutorials/refs/heads/main/rag/aup_config.py

In [ ]:
from aup_config import aup_setup
aup_setup()

## Build the RAG Pipeline

This section explains how to configure and build the RAG pipeline

### Setup Indexing and the Query Engine
Import the necessary libraries

In [ ]:
import requests
import os
import re
import ipywidgets as widgets
import html

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

## RAG System Preparation

### Download Document

We will download a copy of the [Vitis HLS User Guide](https://docs.amd.com/r/en-US/ug1399-vitis-hls) as content for the RAG database.

In [ ]:
base_url = 'https://docs.amd.com/api/khub/maps/JQtJoZLV908LbR5xokDqLw/attachments/9sLLuMlUume6oVQ1~HLSdg-JQtJoZLV908LbR5xokDqLw/content?download=true&Ft-Calling-App=ft%2Fturnkey-portal&Ft-Calling-App-Version=5.2.44'
download_dir = 'data_hls'
pdf_filename = 'vitis_hls_ug.pdf'

os.makedirs(download_dir, exist_ok=True)
if not os.path.isfile(os.path.join(download_dir, pdf_filename)):
    response = requests.get(base_url, stream=True)
    if response.status_code == 200:
        pdf_path = os.path.join(download_dir, pdf_filename)
        with open(pdf_path, 'wb') as file:
            file.write(response.content)

In [ ]:
loader = PyPDFLoader(os.path.join(download_dir, pdf_filename), mode="page")
pdf_doc = loader.load()
print(f"Number of pages in the PDF: {len(pdf_doc)}")

Remove table of contents, and appendices 

In [ ]:
docs = pdf_doc[10:899].copy()
print(f"Number of pages after filtering: {len(docs)}")

Clean up headers and footers

In [ ]:
for doc in docs:
    doc.page_content = doc.page_content.replace('\nSend Feedback', '')
    doc.page_content = re.sub(r'^Vitis HLS User Guide\s*.*$', '',doc.page_content, flags=re.MULTILINE)
    doc.page_content = re.sub(r'^UG1399\s*.*$', '',doc.page_content, flags=re.MULTILINE)

Chunk the document

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=32)
documents_split = text_splitter.split_documents(docs)
print(f"Number of documents after splitting: {len(documents_split)}")

### Embeddings

Instantiate the embedding model that feeds the documents into the vector database

In [ ]:
base_url = "http://localhost:11434/v1/"
api_key = ""
embed_model = OpenAIEmbeddings(model="nomic-embed-text:v1.5", base_url=base_url, api_key=api_key, check_embedding_ctx_length=False)

### Vector Database

Instantiate a Chroma vector database with the document chunks

In [ ]:
vectordb = Chroma.from_documents(documents_split, embed_model)
retriever = vectordb.as_retriever()

## RAG System

In this section, we connect retrieval, prompt formatting, and generation into one LangChain pipeline using **LangChain Expression Language (LCEL)**.

**LCEL basics.** The pipe operator `|` chains steps so the output of one becomes the input of the next, similar to a Unix pipe. For example, `retriever | format_docs` means "call the retriever, then pass its result to `format_docs`."

**`RunnablePassthrough`.** A `RunnablePassthrough()` is a no-op runnable: it receives an input and forwards it unchanged. It is useful inside dictionary steps where some keys need processing (e.g., retrieval) and another key just needs to carry the original input through. Without it, the original input would be consumed by the previous step and unavailable downstream.

In [ ]:
model = ChatOpenAI(model="qwen3.5:9b", base_url=base_url, api_key=api_key, temperature=0.1)

In [ ]:
template = """
Use the following pieces of context to answer the question at the end.

If you don't know the answer, just say that you don't know, don't try to make up an answer.
Use three sentences maximum and keep the answer as concise as possible.

{context}

Question: {question}?

Helpful Answer:
"""

custom_rag_prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

### Baseline LCEL Pipeline

The chain below is read left to right through the `|` operators:

1. **Dictionary step** — builds the two variables that `custom_rag_prompt` expects:
   - `"context"`: runs `retriever | format_docs` — the retriever fetches relevant chunks, then `format_docs` joins them into a single string.
   - `"question"`: uses `RunnablePassthrough()` to forward the original query string unchanged. Without this, the raw query would not be available for the prompt's `{question}` placeholder.
2. **`custom_rag_prompt`** — inserts the `context` and `question` values into the prompt template.
3. **`model`** — sends the filled prompt to the LLM.
4. **`StrOutputParser()`** — extracts the plain-text content from the model's response object.

In [ ]:
rag_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()} |
             custom_rag_prompt | model | StrOutputParser())

In [ ]:
query = "Interfaces Vitis HLS Support"

In [ ]:
rag_chain.invoke(query)

## RAG System + HyDE

### Hypothetical Document Generation

Generate a hypothetical answer document to improve retrieval for difficult queries.
Then use that generated document as the retrieval query so the retriever can find more relevant chunks.

In [ ]:
template = """You are an expert on Vitis HLS.
Write a detailed document on the topic: '{user_query}'.

Your response should be concise and include all key points that would be found in the top search results.
Only focus on the topic, do not explain what Vitis HLS is unless is part of the user question.
"""

prompt_hyde = ChatPromptTemplate.from_template(template)

In [ ]:
generate_docs_for_retrieval = (
    prompt_hyde | model | StrOutputParser()
)

### HyDE Generation Pipeline

`generate_docs_for_retrieval` is a short pipeline that turns the user query into a hypothetical document.
This synthetic document is not the final answer; it is an intermediate text used to improve retrieval.

In [ ]:
hyde_docs = generate_docs_for_retrieval.invoke({'user_query': query})
print(hyde_docs)

`retrieval_chain = generate_docs_for_retrieval | retriever` composes two steps into one flow.
First it generates the hypothetical document, then it uses that text to retrieve supporting chunks from the vector database.

In [ ]:
retrieval_chain = generate_docs_for_retrieval | retriever

Invoke the retriever

In [ ]:
retrieved_docs = retrieval_chain.invoke({'user_query': query})

Let's finally build the RAG + HyDE pipeline where the custom prompt is passed to the model and then we filter only text from the model's response

In [ ]:
rag_hyde_chain = (
    custom_rag_prompt |
    model |
    StrOutputParser()
)

Let's finally invoke the full pipeline

In [ ]:
rag_hyde_chain.invoke({"context": retrieved_docs, "question": query})

### End-to-End HyDE Pipeline

`rag_hyde_chain_full` combines every stage into a single LCEL chain. The key new primitive here is **`RunnablePassthrough.assign()`**: it passes the current dictionary through unchanged *and* adds (or overwrites) a key with the result of a sub-chain. Think of it as "keep everything we have so far, and also compute this new field."

Step-by-step walkthrough of the chain:

1. `{"user_query": RunnablePassthrough()}` — wraps the raw input string into a dict `{"user_query": "<the query>"}` so downstream steps can reference it by name.
2. `RunnablePassthrough.assign(hypothetical_doc=...)` — keeps `user_query` and adds a new key `hypothetical_doc` whose value is produced by running `prompt_hyde | model | StrOutputParser()`.
3. `RunnablePassthrough.assign(context=...)` — keeps everything and adds `context` by retrieving chunks with the hypothetical document and formatting them.
4. `RunnablePassthrough.assign(question=...)` — maps `question` from the original `user_query` so the prompt template can use it.
5. `custom_rag_prompt | model | StrOutputParser()` — fills the prompt, calls the model, and extracts the text.

This pattern is useful when you want one callable chain instead of manually invoking each stage.

In [ ]:
rag_hyde_chain_full = (
    {"user_query": RunnablePassthrough()} |
    RunnablePassthrough.assign(hypothetical_doc=prompt_hyde | model | StrOutputParser()) |
    RunnablePassthrough.assign(context=lambda x: format_docs(retriever.invoke(x["hypothetical_doc"]))) |
    RunnablePassthrough.assign(question=lambda x: x["user_query"]) |
    custom_rag_prompt | 
    model | 
    StrOutputParser()
)

Let's invoke the HyDE pipeline

In [ ]:
rag_hyde_chain_full.invoke("How can pragmas be used in Vitis HLS?")

## Compare RAG vs RAG with HyDE

In [ ]:
control_style = {"description_width": "initial"}

query_input = widgets.Text(
    value="Interfaces support",
    description="Type your Query:",
    layout=widgets.Layout(width="80%"),
    style=control_style,
    placeholder="Ask a question to compare baseline RAG and RAG+HyDE",
)
run_button = widgets.Button(description="Run RAG pipelines", button_style="primary")
status = widgets.HTML()

rag_output = widgets.HTML(
    layout=widgets.Layout(border="1px solid #ddd", padding="8px", width="100%")
)
hyde_output = widgets.HTML(
    layout=widgets.Layout(border="1px solid #ddd", padding="8px", width="100%")
)

def _render_text(panel_widget, title, text):
    safe_text = html.escape(text if text else "No output.")
    panel_widget.value = (
        f"<h4 style='margin:0 0 8px 0;'>{title}</h4>"
        f"<div style='white-space: pre-wrap; overflow-wrap: anywhere; word-break: break-word; max-width:100%; overflow-x:hidden;'>{safe_text}</div>"
    )

def run_comparison(_):
    rag_output.value = ""
    hyde_output.value = ""

    query = query_input.value.strip()
    if not query:
        status.value = "<b>Please enter a query.</b>"
        return

    status.value = f"<b>Query:</b> {html.escape(query)}"

    try:
        rag_answer = rag_chain.invoke(query)
    except Exception as e:
        rag_answer = f"Error running baseline RAG: {e}"

    try:
        hyde_answer = rag_hyde_chain_full.invoke(query)
    except Exception as e:
        hyde_answer = f"Error running RAG+HyDE: {e}"

    _render_text(rag_output, "RAG", rag_answer)
    _render_text(hyde_output, "RAG+HyDE", hyde_answer)

run_button.on_click(run_comparison)

left_panel = widgets.VBox([rag_output], layout=widgets.Layout(width="48%", overflow="hidden", padding="0 4px 0 0"))
right_panel = widgets.VBox([hyde_output], layout=widgets.Layout(width="48%", overflow="hidden", padding="0 4px 0 0"))
button_row = widgets.HBox([run_button], layout=widgets.Layout(width="100%", justify_content="center"))

display(
    widgets.VBox([
        widgets.HTML("<h3>RAG vs RAG+HyDE Output Comparison</h3>"),
        query_input,
        button_row,
        status,
        widgets.HBox([left_panel, right_panel], layout=widgets.Layout(width="100%", justify_content="space-between")),
    ])
)

run_comparison(None)

## Exercises for the Reader

- Change the `chunk_size` and re-run both pipelines. What do you observe?
- Increase/Decrease the number of retrieved documents. What do you observe?
- Rewrite the HyDE prompt in a way that the hypothetical document is written as bullet-points. What do you observe?

## References

<div class="alert alert-block alert-info">
<ul>
    <li><a href="https://arxiv.org/abs/2005.11401">Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks</a></li>
    <li><a href="https://arxiv.org/abs/2212.10496">Precise Zero-Shot Dense Retrieval without Relevance Labels</a></li>
    <li><a href="https://blog.langchain.com/langchain-expression-language/">LangChain Expression Language</a></li>
    <li><a href="https://reference.langchain.com/python/langchain-classic/schema/runnable">LCEL runnable</a></li>
    <li><a href="https://rocm.docs.amd.com/projects/ai-developer-hub/en/v3.1/notebooks/inference/rag_ollama_llamaindex.html">RAG System with LlamaIndex</a></li>
</ul>
</div>

## Conclusions

In this notebook, we explored a traditional RAG system and an enhanced RAG system with Hypothetical Document Embeddings

---

[AMD University Program](https://www.amd.com/aup)

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.

SPDX-License-Identifier: MIT